<a href="https://colab.research.google.com/github/bcdanl/210-code/blob/main/danl_210_class_2026_0506.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classwork 17

In [1]:
import pandas as pd
import numpy as np

# Below is for an interactive display of DataFrame in Colab
from google.colab import data_table
data_table.enable_dataframe_formatter()

cereal = pd.read_csv('https://bcdanl.github.io/data/cereals_oatmeal.csv')

In [2]:
cereal

,Name,Manufacturer,Type,Calories,Fiber,Sugars
0,100% Bran,Nabisco,Cold,70,10.0,6
1,100% Natural Bran,Quaker Oats,Cold,120,2.0,8
2,All-Bran,Kellogg's,Cold,70,9.0,5
3,All-Bran with Extra Fiber,Kellogg's,Cold,50,14.0,0
4,Almond Delight,Ralston Purina,Cold,110,1.0,8
...,...,...,...,...,...,...
72,Triples,General Mills,Cold,110,0.0,3
73,Trix,General Mills,Cold,110,0.0,12
74,Wheat Chex,Ralston Purina,Cold,100,3.0,3
75,Wheaties,General Mills,Cold,100,3.0,3


In [3]:
cereal.columns

Index(['Name', 'Manufacturer', 'Type', 'Calories', 'Fiber', 'Sugars'], dtype='object')

## Q4

Add a Normalized_Sugars variable to the cereal DataFrame that standardizes each cereal’s sugar content within its manufacturer group.



In [ ]:
cereal['Normalized_Sugars'] = cereal.groupby('Manufacturer')['Sugars'].transform( lambda s: ( s - s.mean() ) / s.std()

In [8]:
q4 = (
    cereal
    .assign(
        Normalized_Sugars = cereal.groupby('Manufacturer')['Sugars'].transform(
            lambda s: ( s - s.mean() ) / s.std() ),
        Normalized_Sugars_ungrouped = ( cereal['Sugars'] - cereal['Sugars'].mean() ) / cereal['Sugars'].std()

    )
)
q4

,Name,Manufacturer,Type,Calories,Fiber,Sugars,Normalized_Sugars,Normalized_Sugars_ungrouped
0,100% Bran,Nabisco,Cold,70,10.0,6,1.458030,-0.207447
1,100% Natural Bran,Quaker Oats,Cold,120,2.0,8,0.540062,0.242508
2,All-Bran,Kellogg's,Cold,70,9.0,5,-0.569951,-0.432425
3,All-Bran with Extra Fiber,Kellogg's,Cold,50,14.0,0,-1.680872,-1.557313
4,Almond Delight,Ralston Purina,Cold,110,1.0,8,0.526212,0.242508
...,...,...,...,...,...,...,...,...
72,Triples,General Mills,Cold,110,0.0,3,-1.279350,-0.882380
73,Trix,General Mills,Cold,110,0.0,12,1.044607,1.142419
74,Wheat Chex,Ralston Purina,Cold,100,3.0,3,-0.877019,-0.882380
75,Wheaties,General Mills,Cold,100,3.0,3,-1.279350,-0.882380


In [9]:
# q5

q5 = (
    cereal
    .groupby('Manufacturer')
    .apply(
        lambda df: df.nlargest(2, "Sugars", keep = "all")
    )
)

q5

/tmp/ipykernel_4135/1442716139.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Name  \
Manufacturer                                                        
American Home Food Products 43                              Maypo   
General Mills               70                  Total Raisin Bran   
                            14                        Cocoa Puffs   
                            18                      Count Chocula   
Kellogg's                   66                             Smacks   
                            6                         Apple Jacks   
Nabisco                     0                           100% Bran   
                            68            Strawberry Fruit Wheats   
Post                        30                       Golden Crisp   
                            52              Post Nat. Raisin Bran   
Quaker Oats                 10                       Cap'n'Crunch   
                            35                   Honey Graham Ohs   
Ralston Purina              44   Muesli Raisins; Dates; & Almonds   
                            45  Muesli Raisins; Peaches; & Pecans   

                                               Manufacturer  Type  Calories  \
Manufacturer                                                                  
American Home Food Products 43  American Home Food Products   Hot       100   
General Mills               70                General Mills  Cold       140   
                            14                General Mills  Cold       110   
                            18                General Mills  Cold       110   
Kellogg's                   66                    Kellogg's  Cold       110   
                            6                     Kellogg's  Cold       110   
Nabisco                     0                       Nabisco  Cold        70   
                            68                      Nabisco  Cold        90   
Post                        30                         Post  Cold       100   
                            52                         Post  Cold       120   
Quaker Oats                 10                  Quaker Oats  Cold       120   
                            35                  Quaker Oats  Cold       120   
Ralston Purina              44               Ralston Purina  Cold       150   
                            45               Ralston Purina  Cold       150   

                                Fiber  Sugars  
Manufacturer                                   
American Home Food Products 43    0.0       3  
General Mills               70    4.0      14  
                            14    0.0      13  
                            18    0.0      13  
Kellogg's                   66    1.0      15  
                            6     1.0      14  
Nabisco                     0    10.0       6  
                            68    3.0       5  
Post                        30    0.0      15  
                            52    6.0      14  
Quaker Oats                 10    0.0      12  
                            35    1.0      11  
Ralston Purina              44    3.0      11  
                            45    3.0      11

## Q7

For each manufacturer, choose one “representative cereal” using these rules:

First, calculate the manufacturer’s average calories.

Keep only cereals with calories below the manufacturer’s average.

If a manufacturer has no cereals below its average calories, use all cereals from that manufacturer instead.

Among the remaining cereals, choose the cereal with the highest fiber.

If there is a tie in fiber, choose the cereal with the lowest sugars.

If there is still a tie, choose the cereal with the alphabetically first name.
Return the full row for the selected cereal.


In [11]:
def rep_cereal(df):
  avg = df['Calories'].mean()

  candidate =  df[ df['Calories'] < avg ]

  if len(candidate) == 0:
    candidate = df

  candidate = candidate.sort_values(['Fiber', 'Sugars', 'Name'],
                                    ascending = [False, True, True]).iloc[0]

  return candidate

q7 = (
    cereal
    .groupby('Manufacturer')
    .apply(rep_cereal)
)

q7

/tmp/ipykernel_4135/8887381.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(rep_cereal)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/data_table.py", line 189, in _repr_javascript_module_
    return self._gen_js(self._preprocess_dataframe())
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/colab/data_table.py", line 170, in _preprocess_dataframe
    dataframe = dataframe.reset_index()
                ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py", line 6472, in reset_index
    new_obj.insert(
  File "/usr/local/lib/python3.12/dist-

,Name,Manufacturer,Type,Calories,Fiber,Sugars
Manufacturer,,,,,,
American Home Food Products,Maypo,American Home Food Products,Hot,100,0.0,3
General Mills,Total Whole Grain,General Mills,Cold,100,3.0,3
Kellogg's,All-Bran with Extra Fiber,Kellogg's,Cold,50,14.0,0
Nabisco,100% Bran,Nabisco,Cold,70,10.0,6
Post,Bran Flakes,Post,Cold,90,5.0,5
Quaker Oats,Puffed Wheat,Quaker Oats,Cold,50,1.0,0
Ralston Purina,Bran Chex,Ralston Purina,Cold,90,4.0,6


In [12]:
import pandas as pd
import numpy as np

# Below is for an interactive display of DataFrame in Colab
from google.colab import data_table
data_table.enable_dataframe_formatter()

fortune1000 = pd.read_csv("https://bcdanl.github.io/data/fortune1000_2025.csv")
fortune = fortune1000[[
    "Rank", "Company", "Sector", "Industry",
    "Revenues_M", "Profits_M", "Number_of_Employees"
]]

In [13]:
sector_and_industry = fortune.groupby(["Sector", "Industry"])

sector_and_industry.groups

{('Aerospace & Defense', 'Aerospace & Defense'): [53, 58, 62, 95, 109, 117, 202, 310, 367, 474, 497, 530, 558, 602, 641, 736, 794, 829, 868, 897, 985], ('Apparel', 'Apparel'): [89, 404, 440, 448, 533, 536, 555, 597, 681, 742, 764, 862, 888, 950], ('Business Services', 'Advertising, marketing'): [275, 396, 820, 951], ('Business Services', 'Diversified Outsourcing Services'): [219, 231, 238, 244, 426, 457, 540, 566, 591, 637, 659, 661, 678, 732, 765, 816, 858, 865, 933, 944, 955, 959, 965], ('Business Services', 'Education'): [688, 994, 999], ('Business Services', 'Equipment Leasing'): [980], ('Business Services', 'Financial Data Services'): [126, 136, 151, 178, 207, 304, 401, 412, 516, 549, 601, 673, 686, 738, 751, 755, 778, 780, 867, 940, 946], ('Business Services', 'Miscellaneous'): [754], ('Business Services', 'Specialty Retailers: Other'): [284], ('Business Services', 'Toys, Sporting Goods'): [973], ('Business Services', 'Waste Management'): [196, 265, 585, 646], ('Chemicals', 'Chem

In [14]:

len(sector_and_industry)

83

In [15]:

sector_and_industry.describe()

Rank  \
                                                                  count   
Sector              Industry                                              
Aerospace & Defense Aerospace & Defense                            21.0   
Apparel             Apparel                                        14.0   
Business Services   Advertising, marketing                          4.0   
                    Diversified Outsourcing Services               23.0   
                    Education                                       3.0   
...                                                                 ...   
Wholesalers         Energy                                          1.0   
                    Information Technology Services                 1.0   
                    Wholesalers: Diversified                       20.0   
                    Wholesalers: Electronics and Office Equipment   5.0   
                    Wholesalers: Food and Grocery                   6.0   

                                                                               \
                                                                         mean   
Sector              Industry                                                    
Aerospace & Defense Aerospace & Defense                            466.904762   
Apparel             Apparel                                        607.357143   
Business Services   Advertising, marketing                         611.500000   
                    Diversified Outsourcing Services               650.521739   
                    Education                                      894.666667   
...                                                                       ...   
Wholesalers         Energy                                         244.000000   
                    Information Technology Services                958.000000   
                    Wholesalers: Diversified                       505.550000   
                    Wholesalers: Electronics and Office Equipment  275.600000   
                    Wholesalers: Food and Grocery                  271.333333   

                                                                               \
                                                                          std   
Sector              Industry                                                    
Aerospace & Defense Aerospace & Defense                            314.853760   
Apparel             Apparel                                        229.716081   
Business Services   Advertising, marketing                         325.781624   
                    Diversified Outsourcing Services               252.115030   
                    Education                                      178.130102   
...                                                                       ...   
Wholesalers         Energy                                                NaN   
                    Information Technology Services                       NaN   
                    Wholesalers: Diversified                       239.277965   
                    Wholesalers: Electronics and Office Equipment  337.890219   
                    Wholesalers: Food and Grocery                  293.711877   

                                                                          \
                                                                     min   
Sector              Industry                                               
Aerospace & Defense Aerospace & Defense                             54.0   
Apparel             Apparel                                         90.0   
Business Services   Advertising, marketing                         276.0   
                    Diversified Outsourcing Services               220.0   
                    Education                                      689.0   
...                                                                  ...   
Wholesalers         Energy                                    

In [16]:
agg_tbl = (
    fortune
    .groupby(["Sector", "Industry"])
    .agg(
        n_companies = ("Company",   "size"),
        avg_rev     = ("Revenues_M","mean"),
        tot_rev     = ("Revenues_M","sum")
    )
    .reset_index()  # Flattens the MultiIndex into ordinary columns
)

agg_tbl

,Sector,Industry,n_companies,avg_rev,tot_rev
0,Aerospace & Defense,Aerospace & Defense,21,21551.904762,452590.0
1,Apparel,Apparel,14,9105.714286,127480.0
2,Business Services,"Advertising, marketing",4,8217.500000,32870.0
3,Business Services,Diversified Outsourcing Services,23,7009.130435,161210.0
4,Business Services,Education,3,3380.000000,10140.0
...,...,...,...,...,...
78,Wholesalers,Energy,1,17160.000000,17160.0
79,Wholesalers,Information Technology Services,1,2800.000000,2800.0
80,Wholesalers,Wholesalers: Diversified,20,10285.500000,205710.0
81,Wholesalers,Wholesalers: Electronics and Office Equipment,5,32274.000000,161370.0


In [17]:
fortune['pct_of_industry_rev'] = (
        fortune
        .groupby(["Sector", "Industry"])["Revenues_M"]
        .transform(lambda x: x / x.sum() * 100)
)

fortune

/tmp/ipykernel_4135/3823158045.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fortune['pct_of_industry_rev'] = (


,Rank,Company,Sector,Industry,Revenues_M,Profits_M,Number_of_Employees,pct_of_industry_rev
0,1,Walmart,Retailing,General Merchandisers,680990.0,19440.0,2100000,60.357542
1,2,Amazon,Retailing,Internet Services and Retailing,637960.0,59250.0,1560000,66.306359
2,3,UnitedHealth Group,Health Care,Health Care: Insurance and Managed Care,400280.0,14400.0,400000,43.955416
3,4,Apple,Technology,"Computers, Office Equipment",391040.0,93740.0,164000,63.305812
4,5,CVS Health,Health Care,Health Care: Pharmacy and Other Services,372810.0,4610.0,259500,53.571583
...,...,...,...,...,...,...,...,...
995,996,Datadog,Technology,Computer Software,2680.0,183.7,6500,0.575700
996,997,SBA Communications,Financials,Real estate,2680.0,749.5,1720,1.394816
997,998,Quad/Graphics,Media,"Publishing, Printing",2670.0,-50.9,12200,20.924765
998,999,Americold Realty Trust,Financials,Real estate,2670.0,-94.3,13760,1.389612


In [18]:
def above_median(df):
    med = df["Revenues_M"].median()
    return df[ df["Revenues_M"] > med ]

(
    fortune
    .groupby(["Sector", "Industry"])
    .apply(above_median)
)

/tmp/ipykernel_4135/3816298926.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(above_median)


Rank  \
Sector              Industry                                                  
Aerospace & Defense Aerospace & Defense                           53     54   
                                                                  58     59   
                                                                  62     63   
                                                                  95     96   
                                                                  109   110   
...                                                                     ...   
Wholesalers         Wholesalers: Electronics and Office Equipment 72     73   
                                                                  94     95   
                    Wholesalers: Food and Grocery                 55     56   
                                                                  79     80   
                                                                  121   122   

                                                                                      Company  \
Sector              Industry                                                                    
Aerospace & Defense Aerospace & Defense                           53                      RTX   
                                                                  58          Lockheed Martin   
                                                                  62                   Boeing   
                                                                  95         General Dynamics   
                                                                  109        Northrop Grumman   
...                                                                                       ...   
Wholesalers         Wholesalers: Electronics and Office Equipment 72                TD Synnex   
                                                                  94     Ingram Micro Holding   
                    Wholesalers: Food and Grocery                 55                    Sysco   
                                                                  79   Performance Food Group   
                                                                  121        US Foods Holding   

                                                                                    Sector  \
Sector              Industry                                                                 
Aerospace & Defense Aerospace & Defense                           53   Aerospace & Defense   
                                                                  58   Aerospace & Defense   
                                                                  62   Aerospace & Defense   
                                                                  95   Aerospace & Defense   
                                                                  109  Aerospace & Defense   
...                                                                                    ...   
Wholesalers         Wholesalers: Electronics and Office Equipment 72           Wholesalers   
                                                                  94           Wholesalers   
                    Wholesalers: Food and Grocery                 55           Wholesalers   
                                                                  79           Wholesalers   
                                                                  121          Wholesalers   

                                                                                                            Industry  \
Sector              Industry                                                                                           
Aerospace & Defense Aerospace & Defense                           53                             Aerospace & Defense   
                                                                  58                             Aerospace & Defense   
                                                                  62          